# Module 9: Concurrency & Resource Optimization

**Context**: Avoiding Synapse-style queue bottlenecks in Spark

## Topics
1. Spark's concurrency model (vs Synapse's queueing)
2. Fair Scheduler configuration
3. Dynamic Resource Allocation
4. Concurrent query optimization
5. Workload isolation patterns

In [ ]:
# ── SparkSession: Databricks Connect (remote) / Local fallback ──
from pathlib import Path

try:
    from databricks.connect import DatabricksSession
    spark = DatabricksSession.builder.serverless().getOrCreate()
    MODE = 'databricks'
    S3_RAW = "s3a://sparkling-data-test/data/raw"
    print(f"✅ Databricks Connect | Spark {spark.version}")
except Exception:
    from pyspark.sql import SparkSession
    spark = SparkSession.builder.appName("Module09-Concurrency").master("local[*]").config("spark.sql.shuffle.partitions", "8").config("spark.scheduler.mode", "FAIR").config("spark.dynamicAllocation.enabled", "false").getOrCreate()
    MODE = 'local'
    S3_RAW = None
    print(f"✅ Local Spark {spark.version} | UI: http://localhost:4040")

DATA_RAW = Path("../data/raw")  # local CSV fallback path
print(f"Mode: {MODE}")

---
## 1. Spark vs Synapse: Concurrency Model

### Azure Synapse Problem
- Fixed concurrency slots (e.g., 32 concurrent queries)
- Large queries block smaller ones
- Queue delays during peak hours

### Spark Advantage
- **Task-level parallelism**: Jobs share executor resources
- **Fair Scheduler**: Round-robin between jobs
- **Dynamic Allocation**: Scale executors based on load
- **Pools**: Isolate workloads (interactive vs batch)

In [ ]:
# ── Load data (S3 Parquet or local CSV) ──
transactions = spark.read.parquet(f"{S3_RAW}/transactions") if MODE == "databricks" else spark.read.csv(str(DATA_RAW / "transactions.csv"), header=True, inferSchema=True).cache()

transactions.count()  # Materialize cache
print(f"Loaded {transactions.count():,} transactions")

---
## 2. Concurrent Query Demo

Run multiple queries simultaneously - Spark handles them without queueing.

In [ ]:
def run_query(name, query_func):
    """Run a query and measure time."""
    start = time.time()
    result = query_func()
    elapsed = time.time() - start
    print(f"  {name}: {elapsed:.2f}s")
    return result

# Define different queries
def query_by_channel():
    return transactions.groupBy("channel").agg(count("*"), sum("amount")).collect()

def query_by_status():
    return transactions.groupBy("status").agg(count("*"), avg("amount")).collect()

def query_by_type():
    return transactions.groupBy("txn_type").agg(count("*"), max("amount")).collect()

def query_large_txns():
    return transactions.filter(col("amount") > 50_000_000).count()

In [ ]:
# Sequential execution
print("=== Sequential Execution ===")
start = time.time()
run_query("Channel", query_by_channel)
run_query("Status", query_by_status)
run_query("Type", query_by_type)
run_query("Large", query_large_txns)
print(f"Total sequential: {time.time() - start:.2f}s")

In [ ]:
# Concurrent execution using threads
print("\n=== Concurrent Execution (FAIR Scheduler) ===")
start = time.time()

threads = [
    threading.Thread(target=run_query, args=("Channel", query_by_channel)),
    threading.Thread(target=run_query, args=("Status", query_by_status)),
    threading.Thread(target=run_query, args=("Type", query_by_type)),
    threading.Thread(target=run_query, args=("Large", query_large_txns)),
]

for t in threads:
    t.start()
for t in threads:
    t.join()

print(f"Total concurrent: {time.time() - start:.2f}s")
print("\n💡 Notice: Concurrent is faster because queries share resources!")

---
## 3. Fair Scheduler Pools

Isolate workloads to prevent heavy jobs from starving light queries.

In [ ]:
# Fair Scheduler configuration (fairscheduler.xml)
fair_scheduler_config = """
<?xml version="1.0"?>
<allocations>
  <pool name="production">
    <schedulingMode>FAIR</schedulingMode>
    <weight>2</weight>
    <minShare>4</minShare>
  </pool>
  <pool name="interactive">
    <schedulingMode>FAIR</schedulingMode>
    <weight>3</weight>  <!-- Higher priority -->
    <minShare>2</minShare>
  </pool>
  <pool name="batch">
    <schedulingMode>FIFO</schedulingMode>
    <weight>1</weight>
    <minShare>0</minShare>
  </pool>
</allocations>
"""
print(fair_scheduler_config)

In [ ]:
# Assign job to a pool
spark.sparkContext.setLocalProperty("spark.scheduler.pool", "interactive")

# This query runs in the "interactive" pool (higher priority)
result = transactions.filter(col("status") == "Completed").count()
print(f"Interactive query result: {result:,}")

# Reset to default
spark.sparkContext.setLocalProperty("spark.scheduler.pool", None)

---
## 4. Dynamic Resource Allocation

Auto-scale executors based on workload (essential for shared clusters).

In [ ]:
# Dynamic Allocation configuration (for cluster mode)
dynamic_config = """
# Enable in spark-defaults.conf or SparkSession
spark.dynamicAllocation.enabled=true
spark.dynamicAllocation.minExecutors=2
spark.dynamicAllocation.maxExecutors=20
spark.dynamicAllocation.initialExecutors=4
spark.dynamicAllocation.executorIdleTimeout=60s
spark.dynamicAllocation.schedulerBacklogTimeout=10s

# Required: External Shuffle Service
spark.shuffle.service.enabled=true
"""
print(dynamic_config)

---
## 5. Workload Optimization Patterns

### Pattern 1: Separate Interactive & Batch

In [ ]:
def run_interactive_query(spark, query):
    """Run query with interactive pool priority."""
    spark.sparkContext.setLocalProperty("spark.scheduler.pool", "interactive")
    try:
        return spark.sql(query).collect()
    finally:
        spark.sparkContext.setLocalProperty("spark.scheduler.pool", None)

def run_batch_job(spark, df, output_path):
    """Run batch job with batch pool (lower priority, longer timeout)."""
    spark.sparkContext.setLocalProperty("spark.scheduler.pool", "batch")
    try:
        df.write.mode("overwrite").parquet(output_path)
    finally:
        spark.sparkContext.setLocalProperty("spark.scheduler.pool", None)

### Pattern 2: Query Result Caching

In [ ]:
# Cache frequently accessed aggregations
daily_summary = transactions.withColumn("txn_date", to_date(col("txn_datetime"))) \
    .groupBy("txn_date", "channel") \
    .agg(count("*").alias("txn_count"), sum("amount").alias("total_amount")) \
    .cache()

# Materialize cache
daily_summary.count()
print("✅ Daily summary cached - concurrent queries will be instant")

# Multiple queries on cached data (no re-computation)
print(f"Mobile App total: {daily_summary.filter(col('channel') == 'Mobile App').agg(sum('total_amount')).collect()}")
print(f"Branch total: {daily_summary.filter(col('channel') == 'Branch').agg(sum('total_amount')).collect()}")

### Pattern 3: Broadcast for Dimension Lookups

In [ ]:
# ── Load data (S3 Parquet or local CSV) ──
branches = spark.read.parquet(f"{S3_RAW}/branches") if MODE == "databricks" else spark.read.csv(str(DATA_RAW / "branches.csv"), header=True, inferSchema=True)
accounts = spark.read.parquet(f"{S3_RAW}/accounts") if MODE == "databricks" else spark.read.csv(str(DATA_RAW / "accounts.csv"), header=True, inferSchema=True)

enriched = transactions.join(broadcast(accounts.select("account_id", "branch_id")), "account_id") \
    .join(broadcast(branches.select("branch_id", "region")), "branch_id")
enriched.groupBy("region").agg(sum("amount")).show()

---
## 6. Monitoring Concurrent Queries

### Spark UI Tabs to Watch
- **Jobs**: See all concurrent jobs running
- **Stages**: Identify shuffle-heavy stages
- **Storage**: Check cached DataFrames
- **Executors**: Resource utilization

### Key Metrics
- **Task Time**: Should be balanced across executors
- **Shuffle Read/Write**: High shuffle = query contention
- **GC Time**: >10% indicates memory pressure

---
## Concurrency Best Practices Checklist

✅ Use **FAIR scheduler** for multi-user environments
✅ Configure **scheduler pools** for workload isolation
✅ Enable **Dynamic Allocation** for auto-scaling
✅ **Cache** common aggregations to avoid re-computation
✅ Use **broadcast joins** for dimension lookups
✅ **Partition data** by query access patterns
✅ Monitor **Spark UI** for bottlenecks
✅ Avoid **collect()** on large datasets (brings to driver)

### Avoiding Synapse-Style Queueing
- Spark doesn't have fixed concurrency slots
- Jobs share executor resources dynamically
- Use pools to prioritize interactive queries
- Scale executors with Dynamic Allocation

In [ ]:
transactions.unpersist()
daily_summary.unpersist()
spark.stop()